# AI Shield 03 — inspect mistakes and complementary evidence

Compare errors image by image, review generator/source groups, and display a small verified image gallery.

**Run:** open this notebook in a fresh Colab tab, use a **CPU runtime** (no paid GPU),
then choose **Runtime → Run all** and approve the Drive mount. Each notebook is independent.
It reads the completed experiment `20261026T132144Z_58c9dad3`.

No PyTorch, TensorFlow, model downloads, image staging, or neural-network training.
Colab's standard NumPy, pandas, scikit-learn and Pillow packages are used without installation changes.
The original experiment, checkpoints, thresholds and app stay unchanged.

**Restart recovery:** run all cells again. These are small-table analyses; identical outputs are reused.
An interrupted write is retried. Changed inputs/code/library versions receive a new diagnostic folder.
Results are saved under `MyDrive/AI_Shield_Diagnostics/<training-run>/`.

**Limits:** the test sets have already been inspected. This is exploratory research, not a new blind
benchmark or a deployment decision. It does not test new image inference or JPEG/resize robustness.


## 1. Mount Drive and load the completed run
This cell contains its own analysis code. It does not need the training notebook or any previous cell variables.
The checks verify saved tables and split identities, not every checkpoint or image byte.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, importlib.util, sys, tempfile
from IPython.display import display
import numpy as np
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive')
RUN_NAME = '20261026T132144Z_58c9dad3'

ANALYSIS_SOURCE = '"""CPU-only analysis of a completed run. Never imports or changes image models."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport uuid\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import (roc_auc_score, average_precision_score,\n                             precision_recall_curve, confusion_matrix, brier_score_loss)\n\nVERSION = \'1.0.0\'\nDEFAULT_RUN = \'20261026T132144Z_58c9dad3\'\nSEM = [\'semantic_vit_logit\', \'semantic_vit_prob\', \'semantic_attention_entropy\',\n       \'semantic_attention_focus\', \'semantic_self_attention_entropy\',\n       \'semantic_self_attention_focus\']\nMAIN = [\'effnet_logit\', \'effnet_prob\'] + SEM\nVAE = [\'vae_recon_error\', \'vae_recon_percentile\']\nFACE = [\'face_present\', \'num_faces\', \'largest_face_ratio\', \'face_confidence\',\n        \'openforensics_applicable\', \'openforensics_prob_fake\']\nFULL = MAIN[:2] + [\'effnet_calibrated_prob\'] + VAE + FACE + SEM\nSPLITS = (\'validation\', \'test\', \'unseen_generator\')\n# Fixed before inspecting new test results. No hyperparameter sweep.\nABLATIONS = {\n    \'semantic_features\': SEM,\n    \'two_logits\': [\'effnet_logit\', \'semantic_vit_logit\'],\n    \'two_branches\': MAIN,\n    \'full_without_vae\': [x for x in FULL if x not in VAE],\n    \'full_without_face\': [x for x in FULL if x not in FACE],\n    \'full_refit\': FULL,\n}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for part in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef stable_json(value):\n    return json.dumps(value, sort_keys=True, indent=2, allow_nan=False) + \'\\n\'\n\n\ndef save_bytes(path, data):\n    """Idempotent evidence writes: equal content is reused; changed content is refused."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if path.exists():\n        if path.read_bytes() != data:\n            raise ValueError(f\'Existing diagnostic output differs: {path}. Keep it; use a new analysis version.\')\n        return\n    temp = path.with_name(path.name + \'.tmp-\' + uuid.uuid4().hex)\n    temp.write_bytes(data)\n    temp.replace(path)\n\n\ndef save_json(path, value):\n    save_bytes(path, stable_json(value).encode(\'utf-8\'))\n\n\ndef save_csv(path, frame):\n    save_bytes(path, frame.to_csv(index=False, lineterminator=\'\\n\').encode(\'utf-8\'))\n\n\ndef read_csv(path):\n    # Preserve saved floating-point values, especially scores exactly on a threshold.\n    return pd.read_csv(path, keep_default_na=False, float_precision=\'round_trip\', dtype={\n        \'sample_id\': str, \'sha256\': str, \'group_id\': str, \'prompt_id\': str})\n\n\ndef align(reference, candidate, columns):\n    required = {\'sample_id\', \'label\', *columns}\n    if not required.issubset(candidate):\n        raise ValueError(\'Missing columns: \' + str(sorted(required - set(candidate))))\n    if reference.sample_id.duplicated().any() or candidate.sample_id.duplicated().any():\n        raise ValueError(\'Duplicate sample IDs\')\n    if set(reference.sample_id) != set(candidate.sample_id):\n        raise ValueError(\'Prediction IDs differ from the complete manifest\')\n    ordered = candidate.set_index(\'sample_id\').loc[reference.sample_id].reset_index()\n    if not np.array_equal(reference.label.to_numpy(), ordered.label.to_numpy()):\n        raise ValueError(\'Prediction labels differ from the manifest\')\n    if not np.isfinite(ordered[columns].to_numpy(dtype=float)).all():\n        raise ValueError(\'Non-finite features or scores\')\n    return ordered\n\n\ndef check_partitions(manifests, roles):\n    all_rows = pd.concat(manifests.values(), ignore_index=True)\n    if all_rows.sample_id.duplicated().any():\n        raise ValueError(\'Sample ID leakage across splits\')\n    for col in (\'sha256\', \'group_id\', \'local_path\'):\n        if (all_rows.groupby(col).split.nunique() > 1).any():\n            raise ValueError(col + \' leakage across outer splits\')\n    generated = all_rows[all_rows.label == 1]\n    prompts = generated[generated.prompt_id.astype(str).str.strip().ne(\'\')]\n    if (prompts.groupby(\'prompt_id\').split.nunique() > 1).any():\n        raise ValueError(\'Generated prompt leakage across splits\')\n    aligned = align(manifests[\'train\'], roles, [])\n    if set(aligned.role) != {\'branch_train\', \'calibration\', \'fusion_train\'}:\n        raise ValueError(\'Expected the three original training roles\')\n    for col in (\'sha256\', \'group_id\', \'prompt_id\'):\n        if not np.array_equal(aligned[col].to_numpy(), manifests[\'train\'][col].to_numpy()):\n            raise ValueError(\'Training-role metadata differs: \' + col)\n        chosen = aligned if col != \'prompt_id\' else aligned[\n            (aligned.label == 1) & aligned.prompt_id.astype(str).str.strip().ne(\'\')]\n        if (chosen.groupby(col).role.nunique() > 1).any():\n            raise ValueError(col + \' leakage between fitting roles\')\n    for _, group in aligned.groupby(\'role\'):\n        if set(group.label) != {0, 1}:\n            raise ValueError(\'Each training role must contain both classes\')\n\n\ndef metrics(y, scores, threshold):\n    y = np.asarray(y, dtype=int)\n    scores = np.asarray(scores, dtype=float)\n    if set(y) != {0, 1} or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError(\'Need both classes and finite scores between zero and one\')\n    tn, fp, fn, tp = map(int, confusion_matrix(y, scores >= threshold, labels=[0, 1]).ravel())\n    return {\'n\': len(y), \'auc\': float(roc_auc_score(y, scores)),\n            \'average_precision\': float(average_precision_score(y, scores)),\n            \'accuracy\': (tp + tn) / len(y), \'precision\': tp / max(tp + fp, 1),\n            \'recall\': tp / (tp + fn), \'f1\': 2 * tp / max(2 * tp + fp + fn, 1),\n            \'fpr\': fp / (fp + tn), \'brier\': float(brier_score_loss(y, scores)),\n            \'threshold\': float(threshold), \'tn\': tn, \'fp\': fp, \'fn\': fn, \'tp\': tp}\n\n\ndef best_f1(y, scores):\n    precision, recall, thresholds = precision_recall_curve(y, scores)\n    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-15)\n    return float(thresholds[np.flatnonzero(f1 == f1.max())[-1]])\n\n\ndef low_fpr_threshold(y, scores, target=0.05):\n    """Highest validation recall with FPR <= target; ties use the highest threshold."""\n    y, scores = np.asarray(y), np.asarray(scores, dtype=float)\n    candidates = np.r_[np.unique(scores), np.nextafter(scores.max(), np.inf)]\n    choices = [(float(((scores[y == 1] >= t).mean())), float(t)) for t in candidates\n               if (scores[y == 0] >= t).mean() <= target]\n    return max(choices)[1]\n\n\ndef load_run(root, run_name=DEFAULT_RUN):\n    """Load small tables only. No checkpoint deserialization or image staging."""\n    root = Path(root)\n    if Path(run_name).name != run_name or run_name in (\'.\', \'..\'):\n        raise ValueError(\'Use one saved run folder name\')\n    run = root / \'AI_Shield_Adaptation/full_pipeline_v1\' / run_name\n    cf_root = root / \'AI_Shield_CommunityForensics/experiments/finetune_v1\'\n    split_root = root / \'AI_Shield_Data/splits/v1\'\n    inputs = {}\n\n    def source(path):\n        if not path.is_file():\n            raise FileNotFoundError(f\'Missing saved input: {path}. Mount the Drive account containing the completed run.\')\n        inputs[str(path.relative_to(root)).replace(\'\\\\\', \'/\')] = sha(path)\n        return path\n\n    report = read_json(source(run / \'final_report.json\'))\n    if report.get(\'status\') != \'completed\':\n        raise ValueError(\'These notebooks require the completed training report\')\n    plan = read_json(source(run / \'experiment_plan.json\'))\n    selection = read_json(source(run / \'adapted_full_selection.json\'))\n    lock = read_json(source(run / \'selection_lock.json\'))\n    historical = read_json(source(run / \'historical_reference_hashes.json\'))\n    if plan[\'feature_order\'] != FULL or selection[\'feature_order\'] != FULL:\n        raise ValueError(\'Unexpected full-pipeline feature order\')\n    manifests = {}\n    for split in (\'train\', *SPLITS):\n        p = source(split_root / (split + \'.csv\'))\n        if sha(p) != plan[\'split_sha256\'][split]:\n            raise ValueError(\'Changed original split: \' + split)\n        manifests[split] = read_csv(p)\n        if set(manifests[split].split) != {split} or set(manifests[split].label) != {0, 1}:\n            raise ValueError(\'Bad manifest split/labels: \' + split)\n    roles = read_csv(source(run / \'training_roles.csv\'))\n    for name in (\'training_roles.csv\', \'adapted_full_selection.json\'):\n        if sha(run / name) != lock[\'files\'][name]:\n            raise ValueError(\'Changed locked input: \' + name)\n    check_partitions(manifests, roles)\n    fusion_rows = roles[roles.role == \'fusion_train\']\n    train = align(fusion_rows, read_csv(source(run / \'adapted_full_fusion_training_features.csv\')), FULL)\n    if set(train.split) != {\'train\'}:\n        raise ValueError(\'Fusion features must be training-only\')\n    predictions, cf = {}, {}\n    threshold_path = source(cf_root / \'results/validation_thresholds.json\')\n    cf_threshold = float(read_json(threshold_path)[\'finetuned\'][\'max_f1\'])\n    for split in SPLITS:\n        predictions[split] = align(manifests[split], read_csv(source(\n            run / f\'adapted_full_{split}_predictions.csv\')), FULL + [\'fusion_prob\'])\n        cf_path = source(cf_root / f\'predictions/finetuned_{split}.csv\')\n        cf[split] = align(manifests[split], read_csv(cf_path), [\'prob_fake\'])\n        for path in ([cf_path, threshold_path] if split != \'validation\' else []):\n            suffix = str(path.relative_to(cf_root)).replace(\'\\\\\', \'/\')\n            matches = [v for k, v in historical.items() if k.replace(\'\\\\\', \'/\').endswith(\'/\' + suffix)]\n            if len(matches) != 1 or sha(path) != matches[0]:\n                raise ValueError(\'Changed historical CF input: \' + suffix)\n        if split != \'validation\':\n            for name, scores, threshold in (\n                (\'adapted_full/fusion_prob\', predictions[split].fusion_prob, selection[\'thresholds\'][\'fusion_prob\']),\n                (\'CF/finetuned\', cf[split].prob_fake, cf_threshold)):\n                matches = [x for x in report[\'comparison\'] if x[\'model\'] == name and x[\'split\'] == split]\n                measured = metrics(manifests[split].label, scores, threshold)\n                if len(matches) != 1 or any(not np.isclose(measured[k], matches[0][k], atol=1e-9, rtol=1e-8)\n                                          for k in measured):\n                    raise ValueError(\'Saved comparison cannot be reproduced: \' + name + \'/\' + split)\n    actual_val = metrics(predictions[\'validation\'].label, predictions[\'validation\'].fusion_prob,\n                         selection[\'thresholds\'][\'fusion_prob\'])\n    if any(not np.isclose(actual_val[k], selection[\'validation_metrics\'][k], atol=1e-9, rtol=1e-8)\n           for k in actual_val):\n        raise ValueError(\'Saved validation result differs\')\n    histories = {b: read_csv(source(run / (b + \'_history.csv\'))) for b in (\'effnet\', \'semantic\')}\n    return dict(root=root, run=run, report=report, plan=plan, selection=selection,\n                manifests=manifests, train=train, roles=roles, predictions=predictions,\n                cf=cf, cf_threshold=cf_threshold, histories=histories, input_hashes=inputs)\n\n\ndef output_dir(data, notebook, options=None):\n    versions = {p: importlib.metadata.version(p) for p in (\'numpy\', \'pandas\', \'scikit-learn\', \'Pillow\')}\n    identity = {\'version\': VERSION, \'notebook\': notebook, \'options\': options or {},\n                \'input_hashes\': data[\'input_hashes\'], \'libraries\': versions,\n                \'source_sha256\': sha(__file__)}\n    key = hashlib.sha256(stable_json(identity).encode()).hexdigest()[:16]\n    folder = data[\'root\'] / \'AI_Shield_Diagnostics\' / data[\'run\'].name / (notebook + \'_\' + key)\n    folder.mkdir(parents=True, exist_ok=True)\n    save_json(folder / \'analysis_inputs.json\', identity)\n    print(\'Results:\', folder)\n    return folder\n\n\ndef baseline_scores(data, split):\n    p = data[\'predictions\'][split]\n    return {\'forensic\': p.effnet_prob.to_numpy(), \'semantic\': p.semantic_vit_prob.to_numpy(),\n            \'saved_full\': p.fusion_prob.to_numpy(), \'CF\': data[\'cf\'][split].prob_fake.to_numpy()}\n\n\ndef baseline_thresholds(data):\n    t = data[\'selection\'][\'thresholds\']\n    return {\'forensic\': t[\'effnet_prob\'], \'semantic\': t[\'semantic_vit_prob\'],\n            \'saved_full\': t[\'fusion_prob\'], \'CF\': data[\'cf_threshold\']}\n\n\ndef baseline_table(data):\n    return pd.DataFrame([{\'model\': m, \'split\': split, **metrics(data[\'predictions\'][split].label, s,\n                         baseline_thresholds(data)[m])} for split in SPLITS\n                         for m, s in baseline_scores(data, split).items()])\n\n\ndef score_diagnostics(data):\n    distributions, reliability = [], []\n    for split in SPLITS:\n        y = data[\'predictions\'][split].label.to_numpy()\n        for model, scores in baseline_scores(data, split).items():\n            for label in (0, 1):\n                values = scores[y == label]\n                q = np.quantile(values, [.05, .25, .5, .75, .95])\n                distributions.append(dict(split=split, model=model, label=label, n=len(values),\n                    p05=q[0], p25=q[1], median=q[2], p75=q[3], p95=q[4],\n                    fraction_below_001=float((values < .01).mean()),\n                    fraction_above_099=float((values > .99).mean())))\n            bins = np.minimum((scores * 10).astype(int), 9)\n            for b in range(10):\n                mask = bins == b\n                if not mask.any():\n                    continue\n                reliability.append(dict(split=split, model=model, bin_lower=b / 10, n=int(mask.sum()),\n                    mean_score=float(scores[mask].mean()), actual_generated_fraction=float(y[mask].mean())))\n    return pd.DataFrame(distributions), pd.DataFrame(reliability)\n\n\ndef threshold_diagnostics(data):\n    """Choose alternative thresholds on validation ONLY; freeze before evaluating tests."""\n    y = data[\'predictions\'][\'validation\'].label.to_numpy()\n    chosen = {}\n    for model, scores in baseline_scores(data, \'validation\').items():\n        chosen[model] = {\'saved\': baseline_thresholds(data)[model],\n                         \'validation_fpr_05\': low_fpr_threshold(y, scores)}\n    rows = []\n    for split in SPLITS:\n        for model, scores in baseline_scores(data, split).items():\n            for operating_point, threshold in chosen[model].items():\n                rows.append(dict(model=model, split=split, operating_point=operating_point,\n                    **metrics(data[\'predictions\'][split].label, scores, threshold)))\n    return chosen, pd.DataFrame(rows)\n\n\ndef fit_ablations(data):\n    """Only the reserved fusion TRAIN rows fit weights. No test access here."""\n    train, validation = data[\'train\'], data[\'predictions\'][\'validation\']\n    if set(train.split) != {\'train\'} or set(validation.split) != {\'validation\'}:\n        raise ValueError(\'Wrong fitting/selection split\')\n    expected_ids = set(data[\'roles\'].loc[data[\'roles\'].role == \'fusion_train\', \'sample_id\'])\n    if set(train.sample_id) != expected_ids or set(train.sample_id) & set(validation.sample_id):\n        raise ValueError(\'Fusion fitting IDs are not the reserved training IDs\')\n    models, choices, rows = {}, {}, []\n    for name, columns in ABLATIONS.items():\n        model = make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=5000, random_state=191))\n        model.fit(train[columns], train.label)\n        if int(model[-1].n_iter_.max()) >= model[-1].max_iter:\n            raise RuntimeError(\'Fusion did not converge: \' + name)\n        scores = model.predict_proba(validation[columns])[:, 1]\n        threshold = best_f1(validation.label, scores)\n        choices[name] = {\'columns\': columns, \'threshold\': threshold,\n                         \'validation_metrics\': metrics(validation.label, scores, threshold)}\n        models[name] = model\n        rows.append(dict(model=name, **choices[name][\'validation_metrics\']))\n    # This control must reproduce the saved full fusion; otherwise comparisons are invalid.\n    scores = models[\'full_refit\'].predict_proba(validation[FULL])[:, 1]\n    error = float(np.abs(scores - validation.fusion_prob.to_numpy()).max())\n    if not np.allclose(scores, validation.fusion_prob, atol=5e-4, rtol=1e-4):\n        raise ValueError(f\'Full-refit control differs from saved validation predictions ({error}). Stop; do not relax the check.\')\n    candidates = {name: x[\'validation_metrics\'] for name, x in choices.items()}\n    for name in (\'semantic\', \'forensic\', \'saved_full\'):\n        candidates[name] = metrics(validation.label, baseline_scores(data, \'validation\')[name], baseline_thresholds(data)[name])\n    winner = max(sorted(candidates), key=lambda n: (candidates[n][\'f1\'], candidates[n][\'auc\']))\n    lock = {\'selection_split\': \'validation\', \'rule\': \'highest validation F1, then AUROC, alphabetical tie-break\',\n            \'winner_among_AI_Shield_variants\': winner, \'CF_is_reference_not_selected\': True,\n            \'full_refit_max_validation_error\': error, \'candidates\': choices,\n            \'existing_candidates\': {n: candidates[n] for n in (\'semantic\', \'forensic\', \'saved_full\')},\n            \'fit_sample_ids\': sorted(train.sample_id.tolist()), \'C\': 1., \'seed\': 191,\n            \'test_policy\': \'Historical exploratory comparison; fresh confirmation required before deployment.\'}\n    return models, lock, pd.DataFrame(rows)\n\n\ndef evaluate_ablations(data, models, lock, folder):\n    # Require the pre-test choice to have been written and read back.\n    if read_json(Path(folder) / \'validation_selection.json\') != lock:\n        raise ValueError(\'Save the validation selection before evaluating tests\')\n    rows, predictions = [], {}\n    for split in SPLITS:\n        frame = data[\'predictions\'][split]\n        scores = baseline_scores(data, split)\n        thresholds = baseline_thresholds(data)\n        for name, model in models.items():\n            columns = lock[\'candidates\'][name][\'columns\']\n            scores[name] = model.predict_proba(frame[columns])[:, 1]\n            thresholds[name] = lock[\'candidates\'][name][\'threshold\']\n        result = frame[[\'sample_id\', \'label\', \'split\']].copy()\n        for name, values in scores.items():\n            result[name] = values\n            rows.append(dict(model=name, split=split, **metrics(frame.label, values, thresholds[name])))\n        predictions[split] = result\n        save_csv(Path(folder) / (split + \'_scores.csv\'), result)\n    table = pd.DataFrame(rows)\n    save_csv(Path(folder) / \'ablation_comparison.csv\', table)\n    return table, predictions\n\n\ndef coefficient_table(models):\n    rows = []\n    for name, model in models.items():\n        for feature, weight in zip(ABLATIONS[name], model[-1].coef_[0]):\n            rows.append(dict(model=name, feature=feature, standardized_weight=float(weight)))\n    return pd.DataFrame(rows)\n\n\ndef error_table(data, split):\n    meta = data[\'manifests\'][split]\n    columns = [\'sample_id\', \'label\', \'split\', \'generator\', \'local_path\', \'sha256\']\n    columns += [c for c in (\'source_dataset\', \'source\', \'category\', \'group_id\', \'prompt_id\') if c in meta]\n    result = meta[columns].copy()\n    y = result.label.to_numpy()\n    for name, scores in baseline_scores(data, split).items():\n        result[name + \'_score\'] = scores\n        result[name + \'_prediction\'] = (scores >= baseline_thresholds(data)[name]).astype(int)\n        result[name + \'_correct\'] = result[name + \'_prediction\'].to_numpy() == y\n    full, cf = result.saved_full_correct, result.CF_correct\n    result[\'case\'] = np.select([full & cf, ~full & cf, full & ~cf],\n        [\'both_correct\', \'CF_only_correct\', \'full_only_correct\'], default=\'both_wrong\')\n    result[\'original_extension\'] = result.local_path.map(lambda p: Path(p).suffix.lower())\n    return result\n\n\ndef subgroup_table(errors):\n    """Class-conditional rates; never describe real-only accuracy as AI recall."""\n    rows = []\n    grouping = [\'generator\', \'original_extension\']\n    grouping += [c for c in (\'source_dataset\', \'source\', \'category\') if c in errors]\n    for split, outer in errors.groupby(\'split\'):\n        for field in grouping:\n            for (group, label), frame in outer.groupby([field, \'label\'], dropna=False):\n                for model in (\'forensic\', \'semantic\', \'saved_full\', \'CF\'):\n                    positive = float(frame[model + \'_prediction\'].mean())\n                    rows.append(dict(split=split, field=field, group=str(group), label=int(label), n=len(frame),\n                        model=model, metric=\'generated_recall\' if label == 1 else \'real_false_positive_rate\',\n                        value=positive, small_group=len(frame) < 30))\n    return pd.DataFrame(rows)\n\n\ndef select_gallery(errors, per_case=3):\n    # Deterministic samples from both labels; not a representative prevalence estimate.\n    wanted = errors[errors.case != \'both_correct\'].sort_values(\'sample_id\')\n    return wanted.groupby([\'case\', \'label\'], group_keys=False).head(per_case).reset_index(drop=True)\n\n\ndef gallery_path(original, root):\n    text = str(original).replace(\'\\\\\', \'/\')\n    marker = \'/content/drive/MyDrive/\'\n    return Path(root) / text[len(marker):] if text.startswith(marker) else Path(original)\n\n\ndef render_gallery(selected, root):\n    """Read only selected originals. Missing images yield a labelled placeholder."""\n    from PIL import Image, ImageDraw, ImageOps\n    n = len(selected)\n    if not n:\n        return None, pd.DataFrame(columns=[\'sample_id\', \'status\'])\n    width, height, cols = 330, 300, 3\n    canvas = Image.new(\'RGB\', (cols * width, ((n + cols - 1) // cols) * height), \'white\')\n    draw = ImageDraw.Draw(canvas)\n    checks = []\n    for i, row in enumerate(selected.itertuples()):\n        x, y = (i % cols) * width, (i // cols) * height\n        path = gallery_path(row.local_path, root)\n        status = \'missing_image\'\n        if path.is_file():\n            if sha(path) != row.sha256:\n                status = \'hash_mismatch_not_displayed\'\n            else:\n                try:\n                    with Image.open(path) as im:\n                        im.seek(0)\n                        tile = ImageOps.contain(im.convert(\'RGB\'), (width - 16, 210))\n                    canvas.paste(tile, (x + 8, y + 4))\n                    status = \'verified\'\n                except (OSError, ValueError):\n                    status = \'unreadable_image\'\n        if status != \'verified\':\n            draw.text((x + 8, y + 85), status, fill=\'red\')\n        caption = (f\'{row.sample_id}\\n{row.case} | label={row.label}\\n\'\n                   f\'CF={row.CF_score:.3f} full={row.saved_full_score:.3f}\\n{row.generator}\')\n        draw.text((x + 8, y + 215), caption, fill=\'black\')\n        checks.append(dict(sample_id=row.sample_id, status=status))\n    return canvas, pd.DataFrame(checks)\n\n\ndef interpretation(data, ablations=None):\n    """Evidence-led observations, not an automatic choice of a new deployment model."""\n    base = baseline_table(data)\n    test = base[base.split == \'test\'].set_index(\'model\')\n    messages = [\n        f"Main test: forensic {test.loc[\'forensic\',\'accuracy\']:.1%}, semantic {test.loc[\'semantic\',\'accuracy\']:.1%}, full {test.loc[\'saved_full\',\'accuracy\']:.1%}, CF {test.loc[\'CF\',\'accuracy\']:.1%} accuracy.",\n        \'AUROC compares score ranking across thresholds; low AUROC cannot be repaired by threshold changes alone.\',\n        \'A large gap between real and generated score distributions is desirable. Extreme scores are not proof of calibrated confidence.\',\n        \'These historical tests have already informed research. Any new winner needs a fresh, untouched confirmation set.\',\n        \'This analysis does not test JPEG/resize robustness, new generators, EXIF/C2PA, or historical/context reasoning.\',\n        \'Do not deploy a new threshold or fusion simply because one historical test number increased.\',\n    ]\n    if ablations is not None:\n        val = ablations[ablations.split == \'validation\'].sort_values([\'f1\', \'auc\'], ascending=False)\n        messages.append(\'Validation ranking is saved separately from historical test performance. Inspect false positives as well as F1.\')\n        messages.append(\'Highest validation F1 in the displayed table (including CF reference): \' + str(val.iloc[0][\'model\']))\n    return messages\n'
EXPECTED_SOURCE_SHA256 = '98e1ce765e2f782043e77a8a0c30d911bc89182de830b3c5efaf4f069b5f787e'

assert hashlib.sha256(ANALYSIS_SOURCE.encode()).hexdigest() == EXPECTED_SOURCE_SHA256
code_dir = Path(tempfile.gettempdir()) / ('ai_shield_diagnostics_' + EXPECTED_SOURCE_SHA256[:12])
code_dir.mkdir(parents=True, exist_ok=True)
code_file = code_dir / 'analysis.py'
code_file.write_text(ANALYSIS_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('ai_shield_diagnostics_runtime', code_file)
diag = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = diag
spec.loader.exec_module(diag)
data = diag.load_run(DRIVE_ROOT, RUN_NAME)

OUT = diag.output_dir(data, 'AI_Shield_03_Failure_Review_v1')

print('PASS: saved results reproduced; split identities and training roles verified.')
print('Image models were not loaded. No training will be repeated.')
display(data['roles'].groupby(['role', 'label']).size().rename('images').reset_index())


## 2. Where do CF and the full adapted model disagree?
Use their original saved thresholds. `full_only_correct` means the adapted model is right and CF is wrong;
`CF_only_correct` means the reverse. The ground-truth label is used to describe errors, never to choose a
model for a live upload. Complementary errors do not demonstrate that an ensemble improves accuracy.


In [ ]:
errors = pd.concat([diag.error_table(data, split) for split in ('test', 'unseen_generator')], ignore_index=True)
diag.save_csv(OUT / 'all_error_cases.csv', errors)
summary = errors.groupby(['split', 'case']).size().rename('images').reset_index()
diag.save_csv(OUT / 'disagreement_summary.csv', summary)
display(summary)
for split, frame in errors.groupby('split'):
    sem_corrects = int((frame.semantic_correct & ~frame.CF_correct).sum())
    cf_corrects = int((frame.CF_correct & ~frame.semantic_correct).sum())
    print(split, '| semantic correct / CF wrong:', sem_corrects,
          '| CF correct / semantic wrong:', cf_corrects)


## 3. Which generators and file/source groups cause trouble?
For generated-only groups, report **recall**. For real-only groups, report **false-positive rate**.
Groups with fewer than 30 images are flagged. Filename extension is only a proxy for image encoding.
If label/source/format are strongly associated, the model could exploit shortcuts; this table alone cannot
prove that it does. No automatic artwork/edited-photo labels are invented when the manifest lacks them.


In [ ]:
subgroups = diag.subgroup_table(errors)
diag.save_csv(OUT / 'subgroup_metrics.csv', subgroups)
display(subgroups[(subgroups.field == 'generator') & (subgroups.model.isin(['saved_full', 'CF']))].round(4))
formats = errors.groupby(['split', 'label', 'original_extension']).size().rename('images').reset_index()
diag.save_csv(OUT / 'format_label_counts.csv', formats)
display(formats)
print('Available manifest fields:', ', '.join(data['manifests']['test'].columns))


## 4. Review a small image gallery
Shows up to three examples per error type and true label, separately for each test set.
Selection is deterministic by sample ID, so this is a case review, not a representative sample.
Only these images are opened from Drive. Their SHA-256 is checked before display.
Missing/unreadable/changed images are labelled and do not invalidate the tabular analysis.
The displayed scores are model outputs, not calibrated probabilities of authenticity.


In [ ]:
gallery_checks = []
for split in ('test', 'unseen_generator'):
    selected = diag.select_gallery(errors[errors.split == split], per_case=3)
    diag.save_csv(OUT / (split + '_gallery_selection.csv'), selected)
    canvas, checks = diag.render_gallery(selected, DRIVE_ROOT)
    print(split)
    if canvas is not None:
        display(canvas)
    checks['split'] = split
    gallery_checks.append(checks)
# Availability can change after a Drive remount: use a content-addressed status file.
checks = pd.concat(gallery_checks, ignore_index=True)
check_hash = hashlib.sha256(checks.to_csv(index=False).encode()).hexdigest()[:12]
diag.save_csv(OUT / ('gallery_image_status_' + check_hash + '.csv'), checks)
display(checks.groupby(['split', 'status']).size().rename('images').reset_index())


## 5. Turn observations into a small experiment
The review sheet leaves image category and suspected cause blank for you to annotate. A visual impression is
a hypothesis, not proof of the mechanism. Check whether failures cluster around resizing, unusual lighting,
illustrations, edited real photographs, text, or a particular generator. Do not assume these categories
are covered simply because the benchmark contains many images.

Use notebook 01 to decide whether the issue is score ranking or the operating threshold; use notebook 02
to decide whether components help. Then change one thing, select on validation, and confirm on fresh data.
JPEG/resize inference tests and matched-budget neural-network retraining remain separate future experiments.


In [ ]:
review = errors[errors.case != 'both_correct'].copy()
review['manual_image_category'] = ''
review['suspected_failure_cause'] = ''
review['evidence_or_notes'] = ''
template = OUT / 'manual_review_template.csv'
if not template.exists():
    diag.save_csv(template, review)
else:
    print('Keeping your existing review sheet, including any manual annotations.')
print('Manual review sheet:', template)
for message in diag.interpretation(data):
    print('-', message)


## Completion and next steps
All tables belong to this diagnostic folder; the trained run is read-only.
These findings can motivate a small, predefined experiment. Select models and thresholds using
training/validation only, then confirm once on a fresh test set. Avoid repeatedly tuning against the
historical test results printed here. Any image preprocessing or inference change needs new parity checks.


In [ ]:
diag.save_json(OUT / 'completed.json', {
    'status': 'completed', 'original_run': RUN_NAME,
    'analysis_version': diag.VERSION,
    'neural_network_training_performed': False,
    'deployment_changed': False,
    'scope': 'saved-feature diagnostics; historical tests are exploratory'
})
print('DONE. Results saved to:', OUT)
print('You may disconnect this runtime. Run all again to regenerate the displays.')
